# 04 — Prepare the Power BI Dashboard Layer

**Project:** Olist Brazilian E-Commerce — Delivery Reliability & Customer Experience  
**Purpose:** Build clean, documented tables for a two-page Power BI dashboard without changing KPI definitions.

## Outputs

- `fact_order_dashboard.csv` — one row per order; primary source for overall KPI cards, trend and state analyses.
- `fact_order_category_dashboard.csv` — one row per order-category exposure; for category investigation.
- `fact_order_seller_dashboard.csv` — one row per order-seller exposure; for seller investigation.
- `dim_date.csv`, `dim_state.csv`, `dim_category.csv`, `dim_seller.csv` — dimension tables for Power BI.
- `dashboard_export_validation.csv` — grain and reconciliation checks.
- `dashboard_data_dictionary.csv` — definitions and grains of the exported tables.

**Important:** category and seller facts count *order-segment exposures*, not distinct marketplace orders. An order containing two categories appears once in each category's exposure. Do not add exposure counts across categories or sellers and label the total as unique orders.

Run Notebook 01, Notebook 02 and Notebook 03 first. This notebook reads the analytical model from `data/processed/` and the monthly coverage flags from `outputs/business_analysis/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR = PROJECT_ROOT / "outputs" / "business_analysis"
POWERBI_DIR = PROJECT_ROOT / "outputs" / "powerbi"
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

ORDER_PATH = PROCESSED_DIR / "order_level_analytics.csv"
ITEM_PATH = PROCESSED_DIR / "order_item_analytics.csv"
MONTHLY_PATH = ANALYSIS_DIR / "monthly_delivery_performance.csv"
for required in [ORDER_PATH, ITEM_PATH, MONTHLY_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Missing {required.relative_to(PROJECT_ROOT)}. Run Notebook 01, 02 and 03 first.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Power BI outputs: {POWERBI_DIR.relative_to(PROJECT_ROOT)}")

Project root: C:\Users\62821\Downloads\olist_delivery_analytics_starter (4)\olist_delivery_analytics_starter
Power BI outputs: outputs\powerbi


## 1. Load and validate the source models

Convert CSV boolean fields explicitly, parse timestamps and confirm the expected grain before preparing dashboard tables.

In [2]:
def parse_nullable_bool(series: pd.Series) -> pd.Series:
    """Parse CSV booleans safely while preserving missing values."""
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.astype("boolean")
    normalized = series.astype("string").str.strip().str.lower()
    mapping = {"true": True, "false": False, "1": True, "0": False, "yes": True, "no": False}
    return normalized.map(mapping).astype("boolean")

ORDER_DATES = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
order_dtype = {
    "order_id": "string", "customer_id": "string", "customer_unique_id": "string",
    "order_status": "string", "customer_state": "string", "order_purchase_month": "string",
}
item_dtype = {
    "order_id": "string", "product_id": "string", "seller_id": "string",
    "customer_state": "string", "seller_state": "string", "product_category_label": "string",
}
orders = pd.read_csv(ORDER_PATH, dtype=order_dtype, low_memory=False)
items = pd.read_csv(ITEM_PATH, dtype=item_dtype, low_memory=False)
monthly = pd.read_csv(MONTHLY_PATH, dtype={"purchase_month": "string"})

for col in ORDER_DATES:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")
if "order_purchase_timestamp" in items.columns:
    items["order_purchase_timestamp"] = pd.to_datetime(items["order_purchase_timestamp"], errors="coerce")

for col in ["delivery_metric_eligible", "is_late_delivery", "order_reviewed_flag", "is_low_rated_order", "has_item_records", "has_payment_records", "has_review_records"]:
    if col in orders.columns:
        orders[col] = parse_nullable_bool(orders[col])

if "delivery_metric_eligible" not in orders or "is_late_delivery" not in orders:
    raise KeyError("Expected delivery_metric_eligible and is_late_delivery in the order-level model.")
if not orders["order_id"].notna().all() or not orders["order_id"].is_unique:
    raise AssertionError("Order-level model must contain exactly one non-null row per order_id.")
if not {"order_id", "order_item_id", "price", "freight_value", "seller_id", "product_category_label"}.issubset(items.columns):
    raise KeyError("The order-item model is missing one or more expected columns.")
if items.duplicated(["order_id", "order_item_id"]).any():
    raise AssertionError("Order-item model must be unique by order_id + order_item_id.")

orders["purchase_date"] = orders["order_purchase_timestamp"].dt.normalize()
orders["order_purchase_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype("string")
orders["customer_state"] = orders["customer_state"].fillna("Unknown").astype("string").str.upper()
monthly["include_in_default_trend"] = parse_nullable_bool(monthly["include_in_default_trend"]).fillna(False)
monthly["is_partial_calendar_month"] = parse_nullable_bool(monthly["is_partial_calendar_month"]).fillna(False)
monthly["is_low_volume_source_tail"] = parse_nullable_bool(monthly["is_low_volume_source_tail"]).fillna(False)
monthly["meets_minimum_monthly_volume"] = parse_nullable_bool(monthly["meets_minimum_monthly_volume"]).fillna(False)

print(f"Order fact: {len(orders):,} rows; {orders['order_id'].nunique():,} distinct orders")
print(f"Order-item fact: {len(items):,} rows; {items['order_id'].nunique():,} distinct orders with items")

Order fact: 99,441 rows; 99,441 distinct orders
Order-item fact: 112,650 rows; 98,666 distinct orders with items


## 2. Create the order-level dashboard fact

This table stays at one row per order. Overall order KPIs must use this table, not the expanded item or segment exposure tables.

In [3]:
order_columns = [
    "order_id", "customer_id", "customer_unique_id", "order_status", "order_purchase_timestamp",
    "purchase_date", "order_purchase_month", "customer_state", "delivery_metric_eligible",
    "is_late_delivery", "delivery_outcome", "delivery_delay_days", "purchase_to_delivery_calendar_days",
    "promised_lead_time_calendar_days", "purchase_to_approval_hours", "approval_to_carrier_hours",
    "carrier_to_delivery_hours", "item_count", "distinct_product_count", "distinct_seller_count",
    "distinct_category_count", "item_price_total", "freight_value_total", "item_total_including_freight",
    "payment_value_total", "average_review_score", "review_record_count", "review_score_count",
    "order_reviewed_flag", "is_low_rated_order", "has_item_records", "has_payment_records", "has_review_records",
]
order_columns = [c for c in order_columns if c in orders.columns]
fact_order = orders[order_columns].copy()

# Attach monthly coverage controls from Notebook 03 so sparse source-tail months can be hidden by default.
month_flags = monthly[[c for c in [
    "purchase_month", "include_in_default_trend", "is_partial_calendar_month",
    "is_low_volume_source_tail", "meets_minimum_monthly_volume", "has_approx_99pct_followup"
] if c in monthly.columns]].drop_duplicates("purchase_month")
month_flags = month_flags.rename(columns={"purchase_month": "order_purchase_month"})
fact_order = fact_order.merge(month_flags, how="left", on="order_purchase_month", validate="many_to_one")
for col in ["include_in_default_trend", "is_partial_calendar_month", "is_low_volume_source_tail", "meets_minimum_monthly_volume", "has_approx_99pct_followup"]:
    if col in fact_order.columns:
        fact_order[col] = parse_nullable_bool(fact_order[col]).fillna(False)

# Ensure late values are only interpreted for eligible delivered orders.
invalid_late_flags = fact_order["is_late_delivery"].fillna(False) & ~fact_order["delivery_metric_eligible"].fillna(False)
if invalid_late_flags.any():
    raise AssertionError("Some ineligible orders are marked late; review the model logic before dashboarding.")

## 3. Create category and seller exposure facts

Each row represents an order-category or order-seller combination. Item amounts are aggregated *before* order-level delivery outcomes are attached, preventing item-line fan-out in the dashboard.

In [4]:
item_work = items.copy()
item_work["product_category_label"] = item_work["product_category_label"].fillna("Unknown category").astype("string")
item_work["seller_id"] = item_work["seller_id"].fillna("Unknown seller").astype("string")
for col in ["price", "freight_value"]:
    item_work[col] = pd.to_numeric(item_work[col], errors="coerce")

category_group = item_work.groupby(["order_id", "product_category_label"], dropna=False, sort=False)
fact_category = category_group.agg(
    item_line_count=("order_item_id", "size"),
    distinct_products=("product_id", "nunique"),
    item_price_total=("price", lambda s: s.sum(min_count=1)),
    freight_value_total=("freight_value", lambda s: s.sum(min_count=1)),
).reset_index()

seller_group = item_work.groupby(["order_id", "seller_id"], dropna=False, sort=False)
fact_seller = seller_group.agg(
    item_line_count=("order_item_id", "size"),
    distinct_products=("product_id", "nunique"),
    item_price_total=("price", lambda s: s.sum(min_count=1)),
    freight_value_total=("freight_value", lambda s: s.sum(min_count=1)),
    seller_city=("seller_city", "first") if "seller_city" in item_work.columns else ("seller_id", "first"),
    seller_state=("seller_state", "first") if "seller_state" in item_work.columns else ("seller_id", "first"),
).reset_index()

outcome_cols = [c for c in [
    "order_id", "purchase_date", "order_purchase_month", "customer_state", "order_status",
    "delivery_metric_eligible", "is_late_delivery", "delivery_outcome", "delivery_delay_days",
    "average_review_score", "order_reviewed_flag", "is_low_rated_order", "include_in_default_trend",
    "is_partial_calendar_month", "is_low_volume_source_tail", "meets_minimum_monthly_volume",
] if c in fact_order.columns]
order_outcomes = fact_order[outcome_cols].copy()
fact_category = fact_category.merge(order_outcomes, how="left", on="order_id", validate="many_to_one")
fact_seller = fact_seller.merge(order_outcomes, how="left", on="order_id", validate="many_to_one")

if fact_category.duplicated(["order_id", "product_category_label"]).any():
    raise AssertionError("Category exposure table is not unique by order_id + product_category_label.")
if fact_seller.duplicated(["order_id", "seller_id"]).any():
    raise AssertionError("Seller exposure table is not unique by order_id + seller_id.")

## 4. Create dimensions

Use a shared date and customer-state dimension for consistent slicers. Category and seller dimensions filter their respective exposure facts. Use single-direction, one-to-many relationships in Power BI.

In [5]:
valid_purchase_dates = fact_order["purchase_date"].dropna()
if valid_purchase_dates.empty:
    raise AssertionError("Cannot build date dimension: no valid purchase dates found.")
start_date = valid_purchase_dates.min()
end_date = valid_purchase_dates.max()
dates = pd.date_range(start_date, end_date, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype("int64")
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter_number"] = dim_date["date"].dt.quarter
dim_date["quarter_label"] = "Q" + dim_date["quarter_number"].astype("string")
dim_date["month_number"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%B")
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["year_month_sort"] = dim_date["date"].dt.year * 100 + dim_date["date"].dt.month
dim_date["day_of_month"] = dim_date["date"].dt.day
dim_date["day_name"] = dim_date["date"].dt.strftime("%A")
dim_date["is_weekend"] = dim_date["date"].dt.dayofweek.ge(5)
month_map = month_flags.set_index("order_purchase_month").to_dict("index")
month_key = dim_date["year_month"]
for src, dst in [
    ("include_in_default_trend", "include_in_default_trend"),
    ("is_partial_calendar_month", "is_partial_calendar_month"),
    ("is_low_volume_source_tail", "is_low_volume_source_tail"),
    ("meets_minimum_monthly_volume", "meets_minimum_monthly_volume"),
    ("has_approx_99pct_followup", "has_approx_99pct_followup"),
]:
    dim_date[dst] = month_key.map(lambda m: month_map.get(m, {}).get(src, False)).fillna(False).astype(bool)

dim_state = pd.DataFrame({"customer_state": sorted(fact_order["customer_state"].dropna().astype(str).unique())})
dim_category = pd.DataFrame({"product_category_label": sorted(fact_category["product_category_label"].dropna().astype(str).unique())})
if "seller_city" in fact_seller and "seller_state" in fact_seller:
    dim_seller = (fact_seller[["seller_id", "seller_city", "seller_state"]]
                  .drop_duplicates("seller_id")
                  .sort_values("seller_id")
                  .reset_index(drop=True))
else:
    dim_seller = fact_seller[["seller_id"]].drop_duplicates().sort_values("seller_id").reset_index(drop=True)

## 5. Validate grains, joins and totals before exporting

The segment facts are deliberately non-additive across segment types. We validate financial totals within each segment fact separately against source item lines.

In [6]:
validation_rows = []
def add_check(name, observed, expected, passed, notes):
    validation_rows.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "observed": str(observed),
        "expected": str(expected),
        "notes": notes,
    })

source_orders = int(orders["order_id"].nunique())
add_check("order_fact_one_row_per_order", f"rows={len(fact_order)}, unique={fact_order['order_id'].nunique()}", f"rows={source_orders}, unique={source_orders}", len(fact_order) == source_orders and fact_order["order_id"].is_unique, "Overall KPI source must remain one row per order.")
add_check("category_fact_grain_unique", int(fact_category.duplicated(["order_id", "product_category_label"]).sum()), 0, not fact_category.duplicated(["order_id", "product_category_label"]).any(), "One row per order-category exposure.")
add_check("seller_fact_grain_unique", int(fact_seller.duplicated(["order_id", "seller_id"]).sum()), 0, not fact_seller.duplicated(["order_id", "seller_id"]).any(), "One row per order-seller exposure.")
add_check("date_dimension_unique", int(dim_date["date"].duplicated().sum()), 0, dim_date["date"].is_unique, "One row per calendar date.")
add_check("all_order_purchase_dates_in_date_dimension", int((fact_order["purchase_date"].dropna().isin(dim_date["date"])).sum()), int(fact_order["purchase_date"].notna().sum()), fact_order["purchase_date"].dropna().isin(dim_date["date"]).all(), "All non-null order purchase dates map to the date dimension.")

src_price = pd.to_numeric(items["price"], errors="coerce").sum(min_count=1)
src_freight = pd.to_numeric(items["freight_value"], errors="coerce").sum(min_count=1)
for label, frame in [("order", fact_order), ("category", fact_category), ("seller", fact_seller)]:
    if label == "order":
        observed_price = pd.to_numeric(frame["item_price_total"], errors="coerce").sum(min_count=1)
        observed_freight = pd.to_numeric(frame["freight_value_total"], errors="coerce").sum(min_count=1)
    else:
        observed_price = pd.to_numeric(frame["item_price_total"], errors="coerce").sum(min_count=1)
        observed_freight = pd.to_numeric(frame["freight_value_total"], errors="coerce").sum(min_count=1)
    add_check(f"{label}_fact_item_price_reconciles", f"source={src_price:.6f}; output={observed_price:.6f}; diff={observed_price-src_price:.10f}", "difference within 1e-6", pd.notna(observed_price) and abs(observed_price-src_price) < 1e-6, "Summing within a segment fact should reconcile to item-line price totals.")
    add_check(f"{label}_fact_freight_reconciles", f"source={src_freight:.6f}; output={observed_freight:.6f}; diff={observed_freight-src_freight:.10f}", "difference within 1e-6", pd.notna(observed_freight) and abs(observed_freight-src_freight) < 1e-6, "Summing within a segment fact should reconcile to item-line freight totals.")

eligible_n = int(fact_order["delivery_metric_eligible"].fillna(False).sum())
late_n = int((fact_order["delivery_metric_eligible"].fillna(False) & fact_order["is_late_delivery"].fillna(False)).sum())
add_check("late_orders_subset_of_eligible", f"eligible={eligible_n}; late={late_n}", "late <= eligible", late_n <= eligible_n, "Late delivery numerator must be a subset of its eligible denominator.")
add_check("late_flag_not_set_for_ineligible_orders", int(invalid_late_flags.sum()), 0, int(invalid_late_flags.sum()) == 0, "Do not classify an order as late when its delivery metrics are ineligible.")
validation = pd.DataFrame(validation_rows)
display(validation)
validation.to_csv(POWERBI_DIR / "dashboard_export_validation.csv", index=False)
if validation["status"].ne("PASS").any():
    raise AssertionError("Dashboard export validation failed. Review outputs/powerbi/dashboard_export_validation.csv before loading into Power BI.")

,check,status,observed,expected,notes
0,order_fact_one_row_per_order,PASS,"rows=99441, unique=99441","rows=99441, unique=99441",Overall KPI source must remain one row per order.
1,category_fact_grain_unique,PASS,0,0,One row per order-category exposure.
2,seller_fact_grain_unique,PASS,0,0,One row per order-seller exposure.
3,date_dimension_unique,PASS,0,0,One row per calendar date.
4,all_order_purchase_dates_in_date_dimension,PASS,99441,99441,All non-null order purchase dates map to the d...
5,order_fact_item_price_reconciles,PASS,source=13591643.700000; output=13591643.700000...,difference within 1e-6,Summing within a segment fact should reconcile...
6,order_fact_freight_reconciles,PASS,source=2251909.540000; output=2251909.540000; ...,difference within 1e-6,Summing within a segment fact should reconcile...
7,category_fact_item_price_reconciles,PASS,source=13591643.700000; output=13591643.700000...,difference within 1e-6,Summing within a segment fact should reconcile...
8,category_fact_freight_reconciles,PASS,source=2251909.540000; output=2251909.540000; ...,difference within 1e-6,Summing within a segment fact should reconcile...
9,seller_fact_item_price_reconciles,PASS,source=13591643.700000; output=13591643.700000...,difference within 1e-6,Summing within a segment fact should reconcile...


## 6. Export dashboard tables and a data dictionary

These outputs are designed for import into Power BI. Do not merge the three fact tables directly with one another.

In [7]:
exports = {
    "fact_order_dashboard.csv": fact_order,
    "fact_order_category_dashboard.csv": fact_category,
    "fact_order_seller_dashboard.csv": fact_seller,
    "dim_date.csv": dim_date,
    "dim_state.csv": dim_state,
    "dim_category.csv": dim_category,
    "dim_seller.csv": dim_seller,
}
for filename, frame in exports.items():
    frame.to_csv(POWERBI_DIR / filename, index=False, date_format="%Y-%m-%d %H:%M:%S", float_format="%.6f")
    print(f"Saved {filename}: {len(frame):,} rows x {len(frame.columns)} columns")

# Data dictionary documents table grain and high-risk measure caveats.
dictionary_rows = [
    {"table": "fact_order_dashboard", "grain": "One row per order_id", "use": "Overall KPIs, monthly trends, state analysis", "key": "order_id", "caution": "Use for distinct order counts and order-level rates."},
    {"table": "fact_order_category_dashboard", "grain": "One row per order_id + product_category_label", "use": "Category investigation", "key": "order_id + product_category_label", "caution": "Counts are order-category exposures; an order can appear in multiple categories."},
    {"table": "fact_order_seller_dashboard", "grain": "One row per order_id + seller_id", "use": "Seller investigation", "key": "order_id + seller_id", "caution": "Counts are order-seller exposures; an order can appear under multiple sellers."},
    {"table": "dim_date", "grain": "One row per calendar date", "use": "Date slicing and month ordering", "key": "date", "caution": "Sort year_month by year_month_sort; filter default trend to include_in_default_trend = true."},
    {"table": "dim_state", "grain": "One row per customer state", "use": "Consistent customer-state slicer", "key": "customer_state", "caution": "This is the customer destination state, not seller state."},
    {"table": "dim_category", "grain": "One row per category label", "use": "Category slicer and category chart axis", "key": "product_category_label", "caution": "Unknown/missing category is retained as a category."},
    {"table": "dim_seller", "grain": "One row per seller_id represented in the item data", "use": "Seller investigation", "key": "seller_id", "caution": "Seller results are exposure-based and should use minimum-volume thresholds."},
]
dictionary = pd.DataFrame(dictionary_rows)
dictionary.to_csv(POWERBI_DIR / "dashboard_data_dictionary.csv", index=False)

print("\nDashboard layer complete. Review validation and data dictionary before importing the CSVs into Power BI.")

Saved fact_order_dashboard.csv: 99,441 rows x 38 columns
Saved fact_order_category_dashboard.csv: 99,470 rows x 21 columns
Saved fact_order_seller_dashboard.csv: 100,010 rows x 23 columns
Saved dim_date.csv: 774 rows x 17 columns
Saved dim_state.csv: 27 rows x 1 columns
Saved dim_category.csv: 74 rows x 1 columns
Saved dim_seller.csv: 3,095 rows x 3 columns

Dashboard layer complete. Review validation and data dictionary before importing the CSVs into Power BI.


## 7. Power BI relationship checklist

Create these **one-to-many, single-direction** relationships:

- `dim_date[date]` → `fact_order_dashboard[purchase_date]`
- `dim_date[date]` → `fact_order_category_dashboard[purchase_date]`
- `dim_date[date]` → `fact_order_seller_dashboard[purchase_date]`
- `dim_state[customer_state]` → customer-state field in each of the three facts
- `dim_category[product_category_label]` → `fact_order_category_dashboard[product_category_label]`
- `dim_seller[seller_id]` → `fact_order_seller_dashboard[seller_id]`

Do not create relationships between the fact tables. Hide technical keys from report users, mark `dim_date` as the date table, and sort `dim_date[year_month]` by `dim_date[year_month_sort]`. The DAX measures and page design are documented in `docs/powerbi_dashboard_guide.md` and `sql/powerbi_measures.dax`.